# B1 — What "learning" is

**From:** zero  **To:** training a model with gradient descent you wrote yourself, and knowing what loss, gradient, and learning rate mean forever.

Machine learning in one honest sentence: **adjust numbers until wrongness shrinks.**
- The adjustable numbers are **parameters** (a.k.a. weights).
- The wrongness is the **loss**: one number scoring how bad current predictions are.
- **Training** is the loop that nudges parameters to reduce loss.

Everything else — neural nets, LLMs, the model you ran at Fujitsu — is this sentence with more parameters and fancier bookkeeping. We start with ONE parameter, on real data from our repo.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "rubric.yaml").exists())
import sys
sys.path.insert(0, str(ROOT / "pipeline"))
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(7)
print("ready · repo:", ROOT.name)

import json
turns, dur = [], []
for p in sorted((ROOT / "data" / "normalized").glob("*.json")):
    call = json.loads(p.read_text())
    turns.append(len(call["turns"]))
    dur.append(call["turns"][-1]["end_ms"] / 1000)
x, y = np.array(turns, float), np.array(dur, float)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.scatter(x, y)
ax.set_xlabel("turns in call"); ax.set_ylabel("duration (s)")
ax.set_title("our 11 calls - longer conversations take longer (shocking)"); plt.show()

(How to read a **scatter plot**: each dot is one call; its x is the turn count, its y the duration. A cloud rising to the right = the two grow together.)

## A model with one knob
Propose: `duration ≈ w × turns` — one parameter `w`, "seconds per turn." For any guess of `w` we can score it: **mean squared error**, the average of (prediction − truth)². Squared because: misses in both directions count, big misses hurt disproportionately, and the math stays smooth.

**PREDICT:** plot loss against w from 0 to 12 — what shape must it be, and roughly where is its bottom (eyeball seconds-per-turn from the scatter)?

In [ ]:
def loss(w):
    return ((w * x - y) ** 2).mean()

ws = np.linspace(0, 12, 200)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(ws, [loss(w) for w in ws])
ax.set_xlabel("w (seconds per turn)"); ax.set_ylabel("mean squared error")
ax.set_title("the loss landscape: a valley"); plt.show()
print(f"loss at w=2: {loss(2):.0f}   at w=5: {loss(5):.0f}   at w=9: {loss(9):.0f}")

A valley. Training = walking downhill in it. With one knob we could brute-force every w — but an LLM has billions of knobs, and you cannot grid-search a billion-dimensional valley. We need the *local slope*: which way is downhill from where I stand?

## The one refresher cell (as agreed: quick, then move)
The **derivative** of loss at w answers: *if I nudge w by a tiny ε, how does loss change?* Positive slope → downhill is to the left. For our loss the calculus gives, via the chain rule (outer: square; inner: w·x − y):

d(loss)/dw = mean( 2 · (w·x − y) · x )

Don't take my word — check it empirically with a finite difference (nudge w by 0.001, see what loss does):

In [ ]:
def grad(w):
    return (2 * (w * x - y) * x).mean()

for w in (2.0, 5.0, 9.0):
    eps = 1e-3
    numeric = (loss(w + eps) - loss(w - eps)) / (2 * eps)
    print(f"w={w}: formula {grad(w):10.2f}   nudge-test {numeric:10.2f}")

Formula and nudge-test agree — the calculus is just a fast way to ask "which way is downhill." This *gradient check* trick returns in B4 to keep a whole neural net honest.

## Gradient descent: the loop
Start anywhere. Repeat: compute slope, step *against* it, step size scaled by the **learning rate** (lr).

In [ ]:
w, lr, history = 0.5, 0.0005, []
for step in range(60):
    history.append((w, loss(w)))
    w = w - lr * grad(w)
hist = np.array(history)
print(f"learned w = {w:.2f} s/turn   (closed-form best: {(x @ y) / (x @ x):.2f})")
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].plot(hist[:, 1]); axes[0].set_xlabel("step"); axes[0].set_ylabel("loss"); axes[0].set_title("loss falls as we walk downhill")
axes[1].scatter(x, y); axes[1].plot(x, w * x, color="tab:red", label=f"y = {w:.2f}·x")
axes[1].set_xlabel("turns"); axes[1].set_ylabel("duration (s)"); axes[1].legend(); axes[1].set_title("the fitted model")
plt.tight_layout(); plt.show()

You just trained a model. That falling curve is the same "training loss" curve you watched at Fujitsu — same loop, different scale.

## The learning rate knife edge
**PREDICT each case before running:** lr tiny (0.00005) — what does the loss curve look like? lr good (0.0005)? lr huge (0.0019)?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
for lr, style in [(0.00005, "crawls"), (0.0005, "converges"), (0.0019, "DIVERGES")]:
    w, hs = 0.5, []
    for _ in range(60):
        hs.append(loss(w))
        w = w - lr * grad(w)
    ax.plot(hs, label=f"lr={lr} ({style})")
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.set_yscale("log"); ax.legend()
ax.set_title("learning rate: too small crawls, too big explodes"); plt.show()

The diverging curve is what "my training blew up" means: each overshoot lands on a steeper slope, which orders an even bigger overshoot. Every practitioner war story about "lowering the lr" is this picture.

Vocabulary now yours: **parameter/weight, loss, gradient, learning rate, step, convergence, divergence, training loop.**

## Self-check
1. The one-sentence definition of learning?
2. What does the gradient tell you, in plain words?
3. Why squared error and not absolute error? (Two reasons given above.)
4. Loss reached exactly zero on the training data. Is that automatically good? (Hold your answer — B2 settles it.)
5. **Gotcha:** doubling the learning rate halves training time, a colleague claims. When is that true and when does it detonate?

<details><summary>Answers</summary>

1. Adjust numbers until wrongness shrinks.
2. For each parameter: which direction (and how steeply) the loss changes if you nudge it — i.e., which way is downhill.
3. Penalizes both directions, punishes large misses superlinearly, and is smooth so slopes exist everywhere.
4. Not necessarily — it may have memorized the training points while learning nothing general (overfitting; next book).
5. True while steps stay well inside the valley's curvature; past the critical size, overshoot compounds and loss diverges — the third curve.
</details>